In [ ]:
from functions import query_snowflake_to_df
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score


In [2]:
team = 'LOUCITY'

In [3]:
query = f"""
WITH sampled_users AS (
    SELECT UNIFY_ID
    FROM (
        SELECT DISTINCT UNIFY_ID FROM {team}.MART.MASKED_FCT_MARKETING_ACTIVITIES WHERE UNIFY_ID IS NOT NULL
        UNION
        SELECT DISTINCT UNIFY_ID FROM {team}.MART.MASKED_FCT_TICKET_SALES         WHERE UNIFY_ID IS NOT NULL
        UNION
        SELECT DISTINCT UNIFY_ID FROM {team}.MART.MASKED_FCT_MERCH_TRANSACTIONS   WHERE UNIFY_ID IS NOT NULL
    )
    QUALIFY ROW_NUMBER() OVER (ORDER BY HASH(UNIFY_ID)) <= 10000
),

deduped_tickets AS (
    SELECT
        ORDER_ID,
        MAX(UNIFY_ID)             AS UNIFY_ID,
        MAX(ORDER_DATE)           AS ORDER_DATE,
        SUM(TOTAL_GROSS_SALES)    AS TOTAL_GROSS_SALES,
        SUM(TOTAL_NET_SALES)      AS TOTAL_NET_SALES,
        SUM(TOTAL_DISCOUNT)       AS TOTAL_DISCOUNT,
        SUM(NET_QUANTITY)         AS NET_QUANTITY,
        MAX(IS_PLAN)              AS IS_PLAN,
        MAX(IS_GROUP)             AS IS_GROUP,
        MAX(TICKET_TYPE_CATEGORY) AS TICKET_TYPE_CATEGORY
    FROM {team}.MART.MASKED_FCT_TICKET_SALES
    INNER JOIN sampled_users USING (UNIFY_ID)
    WHERE ORDER_DATE IS NOT NULL
    GROUP BY ORDER_ID
),

deduped_merch AS (
    SELECT
        ORDER_ID,
        MAX(UNIFY_ID)             AS UNIFY_ID,
        MAX(ORDER_DATE)           AS ORDER_DATE,
        SUM(TOTAL_GROSS_SALES)    AS TOTAL_GROSS_SALES,
        SUM(TOTAL_NET_SALES)      AS TOTAL_NET_SALES,
        SUM(TOTAL_DISCOUNT)       AS TOTAL_DISCOUNT,
        SUM(NET_QUANTITY)         AS NET_QUANTITY,
        MAX(PRODUCT_CATEGORY)     AS PRODUCT_CATEGORY
    FROM {team}.MART.MASKED_FCT_MERCH_TRANSACTIONS
    INNER JOIN sampled_users USING (UNIFY_ID)
    WHERE ORDER_DATE IS NOT NULL
    GROUP BY ORDER_ID
),

deduped_mark AS (
    SELECT
        ACTIVITY_ID,
        MAX(UNIFY_ID)              AS UNIFY_ID,
        MAX(COALESCE(ACTIVITY_TIMESTAMP, SEND_TIMESTAMP)) AS ACTIVITY_TIMESTAMP,
        MAX(ACTION_TYPE)           AS ACTION_TYPE
    FROM {team}.MART.MASKED_FCT_MARKETING_ACTIVITIES
    INNER JOIN sampled_users USING (UNIFY_ID)
    WHERE COALESCE(ACTIVITY_TIMESTAMP, SEND_TIMESTAMP) IS NOT NULL
    GROUP BY ACTIVITY_ID
),

raw_timeline AS (
    -- 1. Marketing
    SELECT
        m.UNIFY_ID,
        m.ACTIVITY_TIMESTAMP::TIMESTAMP_NTZ          AS event_timestamp,
        m.ACTION_TYPE                                 AS action_label,
        'marketing'                                   AS activity_type,
        NULL::FLOAT AS merch_total_gross_sales,
        NULL::FLOAT AS merch_total_net_sales,
        NULL::FLOAT AS merch_total_discount,
        NULL::FLOAT AS ticket_total_gross_sales,
        NULL::FLOAT AS ticket_total_net_sales,
        NULL::FLOAT AS ticket_total_discount,
        NULL::FLOAT AS merch_quantity,
        NULL::FLOAT AS ticket_quantity
    FROM deduped_mark m

    UNION ALL

    -- 2. CRM
    SELECT
        c.UNIFY_ID,
        c.ACTIVITY_TIMESTAMP::TIMESTAMP_NTZ           AS event_timestamp,
        LOWER(c.ACTIVITY_TYPE)                        AS action_label,
        'crm'                                         AS activity_type,
        NULL::FLOAT AS merch_total_gross_sales,
        NULL::FLOAT AS merch_total_net_sales,
        NULL::FLOAT AS merch_total_discount,
        NULL::FLOAT AS ticket_total_gross_sales,
        NULL::FLOAT AS ticket_total_net_sales,
        NULL::FLOAT AS ticket_total_discount,
        NULL::FLOAT AS merch_quantity,
        NULL::FLOAT AS ticket_quantity
    FROM {team}.MART.MASKED_FCT_CRM_ACTIVITIES c
    INNER JOIN sampled_users u ON c.UNIFY_ID = u.UNIFY_ID
    WHERE c.ACTIVITY_TIMESTAMP IS NOT NULL

    UNION ALL

    -- 3. Merch
    SELECT
        m.UNIFY_ID,
        m.ORDER_DATE::TIMESTAMP_NTZ                   AS event_timestamp,
        CASE
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%t-shirt%' OR LOWER(m.PRODUCT_CATEGORY) LIKE '%polo%' THEN 'tshirt'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%jersey%'                                             THEN 'jersey'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%hoodie%'                                             THEN 'hoodie'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%sweat%' OR LOWER(m.PRODUCT_CATEGORY) LIKE '%quarter-zip%' THEN 'sweatshirt'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%hat%'                                                THEN 'hat'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%ticket%'                                             THEN 'ticket'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%acces%'                                              THEN 'accessories'
            WHEN LOWER(m.PRODUCT_CATEGORY) LIKE '%scar%'                                               THEN 'scarf'
            ELSE 'misc_merch'
        END                                           AS action_label,
        'merch'                                       AS activity_type,
        m.TOTAL_GROSS_SALES AS merch_total_gross_sales,
        m.TOTAL_NET_SALES    AS merch_total_net_sales,
        m.TOTAL_DISCOUNT     AS merch_total_discount,
        m.NET_QUANTITY       AS merch_quantity,
        NULL::FLOAT          AS ticket_total_gross_sales,
        NULL::FLOAT          AS ticket_total_net_sales,
        NULL::FLOAT          AS ticket_total_discount,
        NULL::FLOAT          AS ticket_quantity
    FROM deduped_merch m

    UNION ALL

    -- 4. Ticket Sales
    SELECT
        t.UNIFY_ID,
        t.ORDER_DATE::TIMESTAMP_NTZ                   AS event_timestamp,
        CASE
            WHEN t.IS_PLAN = 1                                           THEN 'plan'
            WHEN t.IS_GROUP = 1                                          THEN 'group'
            WHEN LOWER(t.TICKET_TYPE_CATEGORY) LIKE '%single%'          THEN 'single'
            WHEN LOWER(t.TICKET_TYPE_CATEGORY) LIKE '%season%'          THEN 'season'
            ELSE 'misc_tickets'
        END                                           AS action_label,
        'ticket'                                      AS activity_type,
        NULL::FLOAT          AS merch_total_gross_sales,
        NULL::FLOAT          AS merch_total_net_sales,
        NULL::FLOAT          AS merch_total_discount,
        NULL::FLOAT          AS merch_quantity,
        t.TOTAL_GROSS_SALES  AS ticket_total_gross_sales,
        t.TOTAL_NET_SALES    AS ticket_total_net_sales,
        t.TOTAL_DISCOUNT     AS ticket_total_discount,
        t.NET_QUANTITY       AS ticket_quantity
    FROM deduped_tickets t
),

next_ticket AS (
    SELECT
        r.*,
        MIN(CASE WHEN activity_type = 'ticket' THEN event_timestamp END)
            OVER (
                PARTITION BY UNIFY_ID
                ORDER BY event_timestamp
                ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
            ) AS next_ticket_purchase_timestamp
    FROM raw_timeline r
)

SELECT
    UNIFY_ID,
    event_timestamp                                         AS action_time,
    activity_type,
    action_label                                            AS action_taken,
    merch_total_gross_sales,
    merch_total_net_sales,
    merch_total_discount,
    merch_quantity,
    ticket_total_gross_sales,
    ticket_total_net_sales,
    ticket_total_discount,
    ticket_quantity,
    next_ticket_purchase_timestamp                          AS converted_at,
    DATEDIFF('day', event_timestamp, next_ticket_purchase_timestamp) AS days_before_purchase
FROM next_ticket
ORDER BY UNIFY_ID, action_time;
"""

In [4]:
data = query_snowflake_to_df(query)

In [5]:
data.columns

Index(['UNIFY_ID', 'ACTION_TIME', 'ACTIVITY_TYPE', 'ACTION_TAKEN',
       'MERCH_TOTAL_GROSS_SALES', 'MERCH_TOTAL_NET_SALES',
       'MERCH_TOTAL_DISCOUNT', 'MERCH_QUANTITY', 'TICKET_TOTAL_GROSS_SALES',
       'TICKET_TOTAL_NET_SALES', 'TICKET_TOTAL_DISCOUNT', 'TICKET_QUANTITY',
       'CONVERTED_AT', 'DAYS_BEFORE_PURCHASE'],
      dtype='object')

In [6]:
data['ticket_purchase'] = (data['TICKET_QUANTITY'] > 0).astype(int)
df = data[['ACTION_TIME', 'UNIFY_ID', 'ACTIVITY_TYPE', 'ACTION_TAKEN', 'ticket_purchase']]
del data

In [7]:
df.head()

,ACTION_TIME,UNIFY_ID,ACTIVITY_TYPE,ACTION_TAKEN,ticket_purchase
0,2024-07-28 01:09:48,0001bd60df64bbde517d093474f542db,marketing,subscribe,0
1,2025-01-23 22:00:00,0001bd60df64bbde517d093474f542db,crm,task,0
2,2025-01-30 22:00:00,0001bd60df64bbde517d093474f542db,crm,task,0
3,2025-02-06 22:00:00,0001bd60df64bbde517d093474f542db,crm,task,0
4,2025-02-13 22:00:00,0001bd60df64bbde517d093474f542db,crm,task,0


In [8]:
df['ACTION_TAKEN'].value_counts()

ACTION_TAKEN
open            740884
click            35295
misc_tickets     17013
subscribe         9959
plan              8616
task              5677
bounce            4251
WEB_VISIT         3143
group             1885
unsubscribe       1866
email             1580
misc_merch         916
call               888
season             759
scarf               66
jersey              14
meeting              8
Name: count, dtype: int64

## making sequences

In [9]:
df["ACTION_TIME"] = pd.to_datetime(df["ACTION_TIME"])

# 1. Sort so sequences are in chronological order per user
df = df.sort_values(["UNIFY_ID", "ACTION_TIME"], kind="stable")

# 2. Map each action to an integer (start at 1, reserve 0 for padding/unknown)
actions = sorted(df["ACTION_TAKEN"].dropna().unique())
action2id = {a: i + 1 for i, a in enumerate(actions)}
id2action = {i: a for a, i in action2id.items()}
df["ACTION_ID"] = df["ACTION_TAKEN"].map(action2id)

# 3. One sequence per user
seqs = (
    df.groupby("UNIFY_ID")["ACTION_ID"]
      .apply(list)
      .rename("ACTION_SEQ")
      .reset_index()
)
seqs["SEQ_LEN"] = seqs["ACTION_SEQ"].str.len()

In [10]:
action2id

{'WEB_VISIT': 1,
 'bounce': 2,
 'call': 3,
 'click': 4,
 'email': 5,
 'group': 6,
 'jersey': 7,
 'meeting': 8,
 'misc_merch': 9,
 'misc_tickets': 10,
 'open': 11,
 'plan': 12,
 'scarf': 13,
 'season': 14,
 'subscribe': 15,
 'task': 16,
 'unsubscribe': 17}

In [11]:
seqs['SEQ_LEN'].describe()

count    10000.000000
mean        83.282000
std        242.099788
min          1.000000
25%          2.000000
50%          6.000000
75%         32.000000
max       6524.000000
Name: SEQ_LEN, dtype: float64

In [12]:
MAX_LEN = 150
def pad(seq, max_len=MAX_LEN):
    seq = seq[-max_len:]                      # keep the most recent actions
    return [0] * (max_len - len(seq)) + seq   # left-pad with 0

X = np.array([pad(s) for s in seqs["ACTION_SEQ"]])

## building the model

In [13]:
WINDOW = 20  # how many past actions the model sees

def make_samples(seqs, window=WINDOW):
    X, y, uids = [], [], []
    for uid, s in zip(seqs["UNIFY_ID"], seqs["ACTION_SEQ"]):
        s = np.asarray(s)
        padded = np.concatenate([np.zeros(window, dtype=int), s])  # left-pad with 0
        for i in range(1, len(s)):            # need at least 1 prior action
            X.append(padded[i : i + window])  # = s[i-window : i], zero-padded
            y.append(s[i])
            uids.append(uid)
    return np.array(X), np.array(y), np.array(uids)

X, y, uids = make_samples(seqs)

In [14]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=uids))
X_train, y_train = X[train_idx], y[train_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

In [15]:
# a) Predict "same as last action"
acc_repeat = (X_test[:, -1] == y_test).mean()

# b) First-order Markov: most common next action given the last action
trans = pd.crosstab(X_train[:, -1], y_train)
best_next = trans.idxmax(axis=1)
most_common = pd.Series(y_train).mode()[0]
pred_markov = pd.Series(X_test[:, -1]).map(best_next).fillna(most_common).astype(int)
acc_markov = (pred_markov.values == y_test).mean()

print(acc_repeat, acc_markov)

0.8854642569201827 0.9050378482486787


In [16]:
V = len(action2id) + 1  # +1 for padding id 0

model = keras.Sequential([
    layers.Embedding(V, 32, mask_zero=True),
    layers.GRU(64),
    layers.Dense(V, activation="softmax"),
])
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)

model.fit(
    X_train, y_train,
    validation_split=0.1, epochs=20, batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
)
model.evaluate(X_test, y_test)

Epoch 1/20
2265/2265 ━━━━━━━━━━━━━━━━━━━━ 79s 34ms/step - accuracy: 0.9196 - loss: 0.3146 - top3: 0.9797 - val_accuracy: 0.9111 - val_loss: 0.3077 - val_top3: 0.9822
Epoch 2/20
2265/2265 ━━━━━━━━━━━━━━━━━━━━ 81s 36ms/step - accuracy: 0.9235 - loss: 0.2819 - top3: 0.9843 - val_accuracy: 0.9140 - val_loss: 0.3009 - val_top3: 0.9837
Epoch 3/20
2265/2265 ━━━━━━━━━━━━━━━━━━━━ 75s 33ms/step - accuracy: 0.9244 - loss: 0.2782 - top3: 0.9849 - val_accuracy: 0.9143 - val_loss: 0.2983 - val_top3: 0.9841
Epoch 4/20
2265/2265 ━━━━━━━━━━━━━━━━━━━━ 74s 33ms/step - accuracy: 0.9250 - loss: 0.2763 - top3: 0.9851 - val_accuracy: 0.9154 - val_loss: 0.2956 - val_top3: 0.9841
Epoch 5/20
2265/2265 ━━━━━━━━━━━━━━━━━━━━ 92s 41ms/step - accuracy: 0.9251 - loss: 0.2752 - top3: 0.9852 - val_accuracy: 0.9150 - val_loss: 0.2952 - val_top3: 0.9846
Epoch 6/20
2265/2265 ━━━━━━━━━━━━━━━━━━━━ 83s 36ms/step - accuracy: 0.9252 - loss: 0.2744 - top3: 0.9852 - val_accuracy: 0.9158 - val_loss: 0.2942 - val_top3: 0.9845
Epoc

[0.28940314054489136, 0.9186374545097351, 0.9861260652542114]

In [18]:
# Baseline 1: repeat last action
acc_repeat = (X_test[:, -1] == y_test).mean()

# Baseline 2: first-order Markov
trans = pd.crosstab(X_train[:, -1], y_train)
best_next = trans.idxmax(axis=1)
fallback = pd.Series(y_train).mode()[0]
pred_markov = pd.Series(X_test[:, -1]).map(best_next).fillna(fallback).astype(int).values
acc_markov = (pred_markov == y_test).mean()

# GRU
probs = model.predict(X_test, batch_size=1024)
pred_gru = probs.argmax(1)
acc_gru = (pred_gru == y_test).mean()

print(f"repeat-last: {acc_repeat:.4f}")
print(f"markov:      {acc_markov:.4f}")
print(f"GRU:         {acc_gru:.4f}")

# Macro-F1: are rare actions being predicted at all?
print("macro-F1 GRU:   ", f1_score(y_test, pred_gru, average="macro"))
print("macro-F1 Markov:", f1_score(y_test, pred_markov, average="macro"))

# Accuracy on the "interesting" cases: where the next action differs from the last
changed = y_test != X_test[:, -1]
print(f"share of steps where action changes: {changed.mean():.3f}")
print(f"GRU acc on those steps:    {(pred_gru[changed] == y_test[changed]).mean():.4f}")
print(f"Markov acc on those steps: {(pred_markov[changed] == y_test[changed]).mean():.4f}")

175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 31ms/step
repeat-last: 0.8855
markov:      0.9050
GRU:         0.9186
macro-F1 GRU:    0.3546532110965765
macro-F1 Markov: 0.27721870533135395
share of steps where action changes: 0.115
GRU acc on those steps:    0.4669
Markov acc on those steps: 0.3244


In [20]:
from sklearn.metrics import classification_report

probs = model.predict(X_test, batch_size=1024)
pred = probs.argmax(1)
ch = y_test != X_test[:, -1]

top3 = np.argsort(probs[ch], axis=1)[:, -3:]
print("top-3 acc on changed steps:", np.mean([y in t for y, t in zip(y_test[ch], top3)]))
print(classification_report(y_test[ch], pred[ch], target_names=None, zero_division=0))

175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step
top-3 acc on changed steps: 0.8801877108080364
              precision    recall  f1-score   support

           1       0.26      0.50      0.34        50
           2       0.01      0.00      0.00       294
           3       0.06      0.01      0.01       120
           4       0.61      0.20      0.30      5535
           5       0.05      0.02      0.03       125
           6       0.00      0.00      0.00       194
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         3
           9       0.09      0.02      0.03        52
          10       0.15      0.04      0.07      1822
          11       0.47      0.85      0.61      9148
          12       0.01      0.00      0.00       998
          13       0.00      0.00      0.00         6
          14       0.00      0.00      0.00       141
          15       0.61      0.47      0.53      1140
          16       0.12      0.05      0.07   

In [21]:
d = df.sort_values(["UNIFY_ID", "ACTION_TIME"], kind="stable").copy()
d["ACTION_ID"] = d["ACTION_TAKEN"].map(action2id)
gap = d.groupby("UNIFY_ID")["ACTION_TIME"].diff().dt.total_seconds().fillna(0)
d["log_gap"] = np.log1p(gap)
d["log_gap"] = (d["log_gap"] - d["log_gap"].mean()) / d["log_gap"].std()

def make_samples(d, window=20):
    Xa, Xt, y, uids = [], [], [], []
    for uid, g in d.groupby("UNIFY_ID"):
        a, t = g["ACTION_ID"].to_numpy(), g["log_gap"].to_numpy()
        a_pad = np.concatenate([np.zeros(window, dtype=int), a])
        t_pad = np.concatenate([np.zeros(window), t])
        for i in range(1, len(a)):
            Xa.append(a_pad[i:i + window])
            Xt.append(t_pad[i:i + window])
            y.append(a[i]); uids.append(uid)
    return np.array(Xa), np.array(Xt)[..., None], np.array(y), np.array(uids)

Xa, Xt, y, uids = make_samples(d)
# reuse train_idx / test_idx from before (same sample order), or re-split by uids

a_in = keras.Input(shape=(20,), dtype="int32")
t_in = keras.Input(shape=(20, 1))
x = layers.Embedding(V, 32, mask_zero=True)(a_in)
x = layers.Concatenate()([x, t_in])
x = layers.GRU(64)(x)
out = layers.Dense(V, activation="softmax")(x)
model2 = keras.Model([a_in, t_in], out)
model2.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])

In [22]:
import numpy as np, pandas as pd
from tensorflow import keras
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report

# Same seed and sample order as before, so the test users should match model 1
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(Xa, y, groups=uids))

# Validation split by user (avoids leakage across users)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
tr_sub, val_sub = next(gss_val.split(Xa[train_idx], y[train_idx], groups=uids[train_idx]))
tr_idx, val_idx = train_idx[tr_sub], train_idx[val_sub]

# Sanity check: confirms the test set is identical to model 1's
assert (y[test_idx] == y_test).all(), "Test sets differ, so the comparison isn't apples to apples"

model2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3")],
)

history = model2.fit(
    [Xa[tr_idx], Xt[tr_idx]], y[tr_idx],
    validation_data=([Xa[val_idx], Xt[val_idx]], y[val_idx]),
    epochs=20, batch_size=256,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
)

Epoch 1/20
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 83s 36ms/step - accuracy: 0.9180 - loss: 0.3147 - top3: 0.9795 - val_accuracy: 0.9312 - val_loss: 0.2736 - val_top3: 0.9838
Epoch 2/20
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 76s 34ms/step - accuracy: 0.9230 - loss: 0.2820 - top3: 0.9843 - val_accuracy: 0.9322 - val_loss: 0.2680 - val_top3: 0.9846
Epoch 3/20
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 75s 33ms/step - accuracy: 0.9244 - loss: 0.2768 - top3: 0.9851 - val_accuracy: 0.9324 - val_loss: 0.2630 - val_top3: 0.9851
Epoch 4/20
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 72s 32ms/step - accuracy: 0.9252 - loss: 0.2740 - top3: 0.9854 - val_accuracy: 0.9331 - val_loss: 0.2617 - val_top3: 0.9851
Epoch 5/20
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 74s 33ms/step - accuracy: 0.9257 - loss: 0.2723 - top3: 0.9855 - val_accuracy: 0.9329 - val_loss: 0.2624 - val_top3: 0.9852
Epoch 6/20
2253/2253 ━━━━━━━━━━━━━━━━━━━━ 78s 35ms/step - accuracy: 0.9260 - loss: 0.2712 - top3: 0.9857 - val_accuracy: 0.9333 - val_loss: 0.2605 - val_top3: 0.9849
Epoc

In [23]:
Xa_test, Xt_test, y_te = Xa[test_idx], Xt[test_idx], y[test_idx]
last = Xa_test[:, -1]
changed = y_te != last

probs1 = model.predict(X_test, batch_size=1024)             # model 1 (actions only)
probs2 = model2.predict([Xa_test, Xt_test], batch_size=1024)  # model 2 (actions + gaps)

# Markov baseline
trans = pd.crosstab(X_train[:, -1], y_train)
best_next = trans.idxmax(axis=1)
fallback = pd.Series(y_train).mode()[0]
pred_markov = pd.Series(last).map(best_next).fillna(fallback).astype(int).values

def summarize(name, pred, probs=None):
    row = {
        "model": name,
        "acc": (pred == y_te).mean(),
        "acc_changed": (pred[changed] == y_te[changed]).mean(),
        "macroF1": f1_score(y_te, pred, average="macro"),
    }
    if probs is not None:
        top3 = np.argsort(probs, axis=1)[:, -3:]
        hit = np.array([t in r for t, r in zip(y_te, top3)])
        row["top3"] = hit.mean()
        row["top3_changed"] = hit[changed].mean()
        p_true = probs[np.arange(len(y_te)), y_te]
        row["logloss"] = -np.log(np.clip(p_true, 1e-9, 1)).mean()
    return row

results = pd.DataFrame([
    summarize("repeat-last", last),
    summarize("markov", pred_markov),
    summarize("GRU (actions)", probs1.argmax(1), probs1),
    summarize("GRU + time gaps", probs2.argmax(1), probs2),
]).set_index("model").round(4)
print(results)

175/175 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step
                    acc  acc_changed  macroF1    top3  top3_changed  logloss
model                                                                       
repeat-last      0.8855       0.0000   0.3792     NaN           NaN      NaN
markov           0.9050       0.3244   0.2772     NaN           NaN      NaN
GRU (actions)    0.9186       0.4669   0.3547  0.9861        0.8802   0.2894
GRU + time gaps  0.9217       0.4772   0.3575  0.9859        0.8796   0.2816


In [24]:
ch = changed
print(classification_report(
    y_te[ch], probs2.argmax(1)[ch],
    labels=sorted(set(y_te[ch])), zero_division=0
))

# Per-class recall on changed steps, model 1 vs model 2
from sklearn.metrics import recall_score
labels = sorted(set(y_te[ch]))
cmp = pd.DataFrame({
    "n": pd.Series(y_te[ch]).value_counts().reindex(labels),
    "recall_m1": recall_score(y_te[ch], probs1.argmax(1)[ch], labels=labels, average=None, zero_division=0),
    "recall_m2": recall_score(y_te[ch], probs2.argmax(1)[ch], labels=labels, average=None, zero_division=0),
}, index=labels)
cmp.index = [id2action.get(i, i) for i in cmp.index]
print(cmp.sort_values("n", ascending=False).round(3))

              precision    recall  f1-score   support

           1       0.27      0.56      0.37        50
           2       0.01      0.00      0.00       294
           3       0.00      0.00      0.00       120
           4       0.72      0.22      0.34      5535
           5       0.00      0.00      0.00       125
           6       0.00      0.00      0.00       194
           7       0.00      0.00      0.00         1
           8       0.00      0.00      0.00         3
           9       0.21      0.06      0.09        52
          10       0.13      0.04      0.06      1822
          11       0.49      0.87      0.63      9148
          12       0.02      0.01      0.02       998
          13       0.00      0.00      0.00         6
          14       0.00      0.00      0.00       141
          15       0.61      0.34      0.44      1140
          16       0.14      0.05      0.08       473
          17       0.00      0.00      0.00       355

    accuracy              